# ClaimsIQ 05 — Compare All Three Frameworks

**This notebook:** Runs the SAME set of cases through Notebook 02's
LangGraph agent, Notebook 03's CrewAI crew, and Notebook 04's SWARM
triage — all three connected to the identical Snowflake MCP server —
and builds a side-by-side comparison table.

### Prerequisite
Notebooks 00 and 01 must have been run already (so the data and the
`mcp_snowflake_server.py` module both exist). This notebook re-defines
lightweight versions of each framework's run function inline, so it can
run standalone without re-executing every cell from 02-04.

## Step 1 — Install & connect

In [ ]:
%pip install -q langgraph crewai openai snowflake-connector-python nest_asyncio

In [ ]:
import os, json
import nest_asyncio
nest_asyncio.apply()

from openai import OpenAI
from mcp_snowflake_server import claims_server, SimpleMCPClient

assert os.environ.get("OPENAI_API_KEY"), "Set OPENAI_API_KEY before continuing"
client = OpenAI()

mcp_client = SimpleMCPClient(claims_server)
mcp_client.connect()
print("Connected. Ready to run all three frameworks.")

## Step 2 — Pick the test cases

Ananya Rao's ambiguous case, plus two contrast cases: pull two more
real claim IDs from your Snowflake CLAIMS table to round out the
comparison with less ambiguous examples.

In [ ]:
from mcp_snowflake_server import get_connection

conn = get_connection()
cs = conn.cursor()
cs.execute("SELECT claim_id, order_id FROM CLAIMS WHERE claim_id != 'CLM99001' ORDER BY RANDOM() LIMIT 2")
extra_claims = cs.fetchall()
cs.execute("SELECT o.customer_id FROM ORDERS o WHERE o.order_id = %s", (extra_claims[0][1],))
extra_customer_1 = cs.fetchone()[0]
cs.execute("SELECT o.customer_id FROM ORDERS o WHERE o.order_id = %s", (extra_claims[1][1],))
extra_customer_2 = cs.fetchone()[0]
conn.close()

TEST_CASES = [
    {"label": "Ananya Rao (ambiguous)", "customer_id": "CUST99001", "claim_id": "CLM99001"},
    {"label": "Random case 1", "customer_id": extra_customer_1, "claim_id": extra_claims[0][0]},
    {"label": "Random case 2", "customer_id": extra_customer_2, "claim_id": extra_claims[1][0]},
]
for t in TEST_CASES:
    print(t)

## Step 3 — A minimal LangGraph runner (same logic as Notebook 02)

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, END

class ClaimState(TypedDict):
    customer_id: str
    claim_id: str
    profile: dict
    claim_details: dict
    fraud_signals: list
    velocity: dict
    peer_comparison: dict
    decision: str
    reasoning: str

def lg_executor(state):
    profile = mcp_client.call_tool("get_customer_profile", customer_id=state["customer_id"])
    claim = mcp_client.call_tool("get_claim_details", claim_id=state["claim_id"])
    signals = mcp_client.call_tool("check_fraud_signals", customer_id=state["customer_id"])
    velocity = mcp_client.call_tool("get_transaction_velocity", customer_id=state["customer_id"])
    peer = mcp_client.call_tool("compare_to_peer_spend", customer_id=state["customer_id"])
    return {"profile": profile, "claim_details": claim, "fraud_signals": signals, "velocity": velocity, "peer_comparison": peer}

def lg_validator(state):
    prompt = f"""Synthesize this evidence and decide APPROVED, DENIED, or ESCALATED.
PROFILE: {state['profile']}
CLAIM: {state['claim_details']}
FRAUD SIGNALS: {state['fraud_signals']}
VELOCITY: {state['velocity']}
PEER COMPARISON: {state['peer_comparison']}
Respond: DECISION: <value>  then REASONING: <1-2 sentences>"""
    resp = client.chat.completions.create(model="gpt-4o-mini", messages=[{"role": "user", "content": prompt}], temperature=0)
    text = resp.choices[0].message.content
    decision = "ESCALATED"
    for line in text.split("\n"):
        if line.startswith("DECISION:"):
            decision = line.replace("DECISION:", "").strip()
    return {"decision": decision, "reasoning": text}

lg = StateGraph(ClaimState)
lg.add_node("executor", lg_executor)
lg.add_node("validator", lg_validator)
lg.set_entry_point("executor")
lg.add_edge("executor", "validator")
lg.add_edge("validator", END)
langgraph_app = lg.compile()

def run_langgraph(customer_id, claim_id):
    result = langgraph_app.invoke({"customer_id": customer_id, "claim_id": claim_id})
    return result["decision"], result["reasoning"]

print("LangGraph runner ready.")

## Step 4 — A minimal CrewAI runner (same logic as Notebook 03)

In [ ]:
from crewai import Agent, Task, Crew, Process
from crewai.tools import tool

@tool("Get Customer Profile")
def cr_profile(customer_id: str) -> str:
    """Looks up customer profile and recent orders."""
    return str(mcp_client.call_tool("get_customer_profile", customer_id=customer_id))

@tool("Get Claim Details")
def cr_claim(claim_id: str) -> str:
    """Looks up claim details joined with its order."""
    return str(mcp_client.call_tool("get_claim_details", claim_id=claim_id))

@tool("Assess Fraud Risk")
def cr_fraud(customer_id: str) -> str:
    """Checks fraud signals, transaction velocity, and peer spend comparison together."""
    return str({
        "signals": mcp_client.call_tool("check_fraud_signals", customer_id=customer_id),
        "velocity": mcp_client.call_tool("get_transaction_velocity", customer_id=customer_id),
        "peer": mcp_client.call_tool("compare_to_peer_spend", customer_id=customer_id),
    })

cr_intake = Agent(role="Intake Specialist", goal="Gather profile and claim details", backstory="Detail-oriented.", tools=[cr_profile, cr_claim], verbose=False)
cr_risk = Agent(role="Risk Analyst", goal="Assess fraud risk", backstory="Distinguishes personal risk from market trends.", tools=[cr_fraud], verbose=False)
cr_approver = Agent(role="Approver", goal="Decide APPROVED, DENIED, or ESCALATED", backstory="Weighs claim plausibility against account risk.", verbose=False)

def run_crewai(customer_id, claim_id):
    t1 = Task(description=f"Gather profile for {customer_id} and claim details for {claim_id}.", expected_output="Profile and claim summary.", agent=cr_intake)
    t2 = Task(description=f"Assess fraud risk for {customer_id}.", expected_output="Risk assessment.", agent=cr_risk, context=[t1])
    t3 = Task(description="Decide APPROVED, DENIED, or ESCALATED with reasoning.", expected_output="Decision + reasoning.", agent=cr_approver, context=[t1, t2])
    crew = Crew(agents=[cr_intake, cr_risk, cr_approver], tasks=[t1, t2, t3], process=Process.sequential, verbose=False)
    return str(crew.kickoff())

print("CrewAI runner ready.")

## Step 5 — A minimal SWARM runner (same logic as Notebook 04)

In [ ]:
SW_TRIAGE = """Call get_transaction_velocity. If unrecognized_device_txns >= 2, call transfer_to_specialist. Otherwise approve directly."""
SW_SPECIALIST = """Call check_fraud_signals and compare_to_peer_spend. Use customer_to_peer_ratio as given. If ratio > 3 AND a HIGH signal exists, decide high risk directly. If ratio < 1.5, decide low risk directly. Otherwise call transfer_to_escalation."""
SW_ESCALATION = """Summarize the case and state it is flagged for human review."""

sw_tools = [
    {"type": "function", "function": {"name": "get_transaction_velocity", "description": "Velocity + unrecognized device count.", "parameters": {"type": "object", "properties": {"customer_id": {"type": "string"}}, "required": ["customer_id"]}}},
    {"type": "function", "function": {"name": "check_fraud_signals", "description": "Fraud signals, last 30 days.", "parameters": {"type": "object", "properties": {"customer_id": {"type": "string"}}, "required": ["customer_id"]}}},
    {"type": "function", "function": {"name": "compare_to_peer_spend", "description": "customer_to_peer_ratio.", "parameters": {"type": "object", "properties": {"customer_id": {"type": "string"}}, "required": ["customer_id"]}}},
    {"type": "function", "function": {"name": "transfer_to_specialist", "description": "Hand off to Fraud Specialist.", "parameters": {"type": "object", "properties": {}}}},
    {"type": "function", "function": {"name": "transfer_to_escalation", "description": "Hand off to Escalation.", "parameters": {"type": "object", "properties": {}}}},
]
SW_HANDOFFS = {"transfer_to_specialist": "specialist", "transfer_to_escalation": "escalation"}
SW_INSTR = {"triage": SW_TRIAGE, "specialist": SW_SPECIALIST, "escalation": SW_ESCALATION}
SW_FUNCS = {
    "get_transaction_velocity": lambda customer_id: mcp_client.call_tool("get_transaction_velocity", customer_id=customer_id),
    "check_fraud_signals": lambda customer_id: mcp_client.call_tool("check_fraud_signals", customer_id=customer_id),
    "compare_to_peer_spend": lambda customer_id: mcp_client.call_tool("compare_to_peer_spend", customer_id=customer_id),
}

def run_swarm(customer_id, max_hops=6):
    agent = "triage"
    messages = [{"role": "system", "content": SW_INSTR[agent]}, {"role": "user", "content": f"Evaluate customer {customer_id}."}]
    for _ in range(max_hops):
        resp = client.chat.completions.create(model="gpt-4o-mini", messages=messages, tools=sw_tools, temperature=0)
        msg = resp.choices[0].message
        if not msg.tool_calls:
            return msg.content
        messages.append(msg)
        for tc in msg.tool_calls:
            args = json.loads(tc.function.arguments)
            name = tc.function.name
            if name in SW_HANDOFFS:
                agent = SW_HANDOFFS[name]
                messages.append({"role": "tool", "tool_call_id": tc.id, "content": f"Transferred to {agent}."})
                messages[0] = {"role": "system", "content": SW_INSTR[agent]}
            else:
                result = SW_FUNCS[name](**args)
                messages.append({"role": "tool", "tool_call_id": tc.id, "content": json.dumps(result, default=str)})
    return "Max hops reached."

print("SWARM runner ready.")

## Step 6 — Run all three frameworks against every test case

In [ ]:
comparison = []
for case in TEST_CASES:
    print(f"\n{'='*70}\nRunning: {case['label']}\n{'='*70}")

    lg_decision, lg_reasoning = run_langgraph(case["customer_id"], case["claim_id"])
    print(f"LangGraph: {lg_decision}")

    cr_result = run_crewai(case["customer_id"], case["claim_id"])
    print(f"CrewAI: {cr_result[:150]}...")

    sw_result = run_swarm(case["customer_id"])
    print(f"SWARM: {sw_result[:150]}...")

    comparison.append({
        "case": case["label"],
        "langgraph": lg_decision,
        "crewai_summary": cr_result[:200],
        "swarm_summary": sw_result[:200],
    })

## Step 7 — Build the comparison table

In [ ]:
import pandas as pd

df = pd.DataFrame(comparison)
pd.set_option("display.max_colwidth", 80)
print(df.to_string(index=False))

## Deliverable

1. The full comparison table from Step 7, plus the un-truncated
   reasoning for each framework on Ananya Rao's case specifically
   (re-print `comparison[0]` in full, not truncated to 200 characters).
2. Did all three frameworks reach a similar LEVEL of concern for Ananya
   Rao (whether or not they used identical labels)? Where did they
   diverge, and can you trace the divergence back to a specific tool
   call one framework made that another didn't?
3. Final write-up (the actual capstone deliverable from the project
   brief): based on everything you've now built and compared, which
   framework would you recommend ShopEase actually deploy for real
   fraud triage — and would your answer change if the deciding factor
   were accuracy vs. explainability vs. engineering time to build?